# Lab 3: Deploying Player Segmentation Agent to Amazon Bedrock AgentCore Runtime
## Deploy your RAG-enhanced agent to a production-ready, serverless environment

## Overview

In this lab, you'll learn how to deploy your existing player segmentation agent to Amazon Bedrock AgentCore Runtime. This provides a production-ready, serverless environment for running AI agents at scale.

### What You'll Build

**Production Agent Deployment** with:

- **Serverless Runtime**: Deploy agent to Amazon Bedrock AgentCore with auto-scaling
- **Session Isolation**: Dedicated microVMs for secure, isolated agent execution
- **Production Infrastructure**: IAM roles, ECR repositories, and monitoring setup
- **Multi-framework Support**: Compatible with Strands Agents, LangChain, LangGraph, CrewAI
- **Built-in Observability**: CloudWatch integration for monitoring and debugging
- **Pay-per-use Model**: Cost-effective scaling with usage-based pricing

## Prerequisites

To execute this lab you will need:
* AWS credentials configured
* Amazon Bedrock AgentCore access
* Completed Lab 2 (player_segmentation_agent.py file)

> **How to run:** Click each code cell and press Shift+Enter or the Run button. Execute cells in order from top to bottom.

## Step 1: Install Dependencies

First, let's install the required packages for SageMaker environment.

In [ ]:
# Install required packages for SageMaker
!pip install strands-agents boto3 pandas matplotlib seaborn plotly --q
print("✅ Dependencies installed successfully")

## Step 2: Environment Setup

Configure the development environment and verify AWS credentials for AgentCore deployment. This establishes the necessary connections and validates permissions required for deploying agents to the serverless runtime environment.

In [ ]:
import sys
import os
import json
import time
from datetime import datetime
import boto3

print("✅ Environment setup complete")

# Check AWS credentials
session = boto3.Session()
region = session.region_name or 'us-west-2'
print(f"🌍 AWS Region: {region}")
print("✅ AWS credentials configured")

## Step 3: Prepare Agent for AgentCore Runtime

Create an AgentCore-compatible wrapper for our player segmentation agent from Lab 2. [Amazon Bedrock AgentCore Runtime](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/agents-tools-runtime.html) is a secure, serverless runtime purpose-built for deploying and scaling dynamic AI agents using any open-source framework including Strands Agents. It provides extended runtime support, fast cold starts, true session isolation, and built-in identity management.

This wrapper implements the BedrockAgentCoreApp interface, enabling the agent to run in the serverless runtime environment with proper payload handling.

In [ ]:
%%writefile agentcore_app_player_segmentation_agent.py
from bedrock_agentcore.runtime import BedrockAgentCoreApp
import sys
import os

sys.path.append(os.path.dirname(os.path.abspath(__file__)))
from player_segmentation_agent import create_segmentation_agent

app = BedrockAgentCoreApp()
agent = create_segmentation_agent()

@app.entrypoint
def player_segmentation_agent(payload):
    # Handle health checks
    user_input = payload.get("prompt", "")
    if user_input.lower() == "ping":
        return {"status": "healthy"}
    
    # Handle normal requests
    if not user_input:
        user_input = "Analyze player segments"
    response = agent(user_input)
    return {"result": response.message}

if __name__ == "__main__":
    app.run()

## Step 4: Create Requirements File

Define the Python dependencies needed for our agent to run in the AgentCore Runtime environment. This includes the Strands framework, AWS SDKs, and data processing libraries required for vector operations and player analytics.

In [ ]:
%%writefile requirements.txt
strands-agents>=1.0.0
bedrock-agentcore
bedrock-agentcore-starter-toolkit
boto3>=1.26.0
pandas>=1.3.0
numpy>=1.21.0
matplotlib>=3.4.0

### Install Dependencies

Install the AgentCore requirements to ensure all dependencies are available in the current environment.

In [ ]:
# Install AgentCore requirements
!pip install -r requirements.txt --q

## Step 5: Create IAM Role for AgentCore Runtime

Create a comprehensive IAM role that grants the AgentCore Runtime necessary permissions to access AWS services on behalf of our agent. This includes S3 Vectors access, Bedrock model invocation, ECR image pulling, and SSM parameter retrieval for secure, least-privilege operation.

In [ ]:
import boto3
import json

def create_agentcore_role(agent_name):
    """Create IAM role for AgentCore Runtime with S3 Vectors permissions."""
    iam = boto3.client('iam')
    
    role_name = f"{agent_name}-agentcore-role"
    
    trust_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Principal": {
                    "Service": "bedrock-agentcore.amazonaws.com"
                },
                "Action": "sts:AssumeRole"
            }
        ]
    }

    try:
        role = iam.create_role(
            RoleName=role_name,
            AssumeRolePolicyDocument=json.dumps(trust_policy),
            Description=f"IAM role for {agent_name} AgentCore Runtime"
        )
        
        # Attach Bedrock permissions
        iam.attach_role_policy(
            RoleName=role_name,
            PolicyArn="arn:aws:iam::aws:policy/AmazonBedrockFullAccess"
        )
        
        # Create inline policy for S3 Vectors
        iam.put_role_policy(
            RoleName=role_name,
            PolicyName="S3VectorsAccess",
            PolicyDocument=json.dumps({
                "Version": "2012-10-17",
                "Statement": [{"Effect": "Allow", "Action": ["s3vectors:*"], "Resource": "*"}]
            })
        )
        
        # Add ECR permissions
        iam.put_role_policy(
            RoleName=role_name,
            PolicyName="ECRAccess",
            PolicyDocument=json.dumps({
                "Version": "2012-10-17",
                "Statement": [{
                    "Effect": "Allow", 
                    "Action": [
                        "ecr:GetAuthorizationToken",
                        "ecr:BatchGetImage", 
                        "ecr:GetDownloadUrlForLayer"
                    ], 
                    "Resource": "*"
                }]
            })
        )
        
        # Add SSM permissions
        iam.put_role_policy(
            RoleName=role_name,
            PolicyName="SSMParameterAccess",
            PolicyDocument=json.dumps({
                "Version": "2012-10-17",
                "Statement": [{
                    "Effect": "Allow",
                    "Action": [
                        "ssm:GetParameter",
                        "ssm:GetParameters"
                    ],
                    "Resource": "arn:aws:ssm:*:*:parameter/workshop/*"
                }]
            })
        )
        
        # Add S3 permissions
        iam.put_role_policy(
            RoleName=role_name,
            PolicyName="S3DataAccess",
            PolicyDocument=json.dumps({
                "Version": "2012-10-17",
                "Statement": [{
                    "Effect": "Allow",
                    "Action": [
                        "s3:GetObject",
                        "s3:ListBucket",
                        "s3:PutObject",
                        "s3:PutObjectAcl"
                    ],
                    "Resource": [
                        "arn:aws:s3:::games-data-*",
                        "arn:aws:s3:::games-data-*/*"
                    ]
                }]
            })
        )
        
        print(f"✅ Created IAM role: {role_name}")
        return role
        
    except iam.exceptions.EntityAlreadyExistsException:
        role = iam.get_role(RoleName=role_name)
        print(f"✅ Using existing IAM role: {role_name}")
        return role

agent_name = "player_segmentation_agent"
agentcore_iam_role = create_agentcore_role(agent_name=agent_name)
print(f"Role ARN: {agentcore_iam_role['Role']['Arn']}")

## Step 6: Configure AgentCore Runtime Deployment

Configure the AgentCore Runtime deployment settings using the Strands toolkit. This setup defines the agent entrypoint, execution role, ECR repository creation, and regional deployment parameters needed for serverless agent hosting.

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

agentcore_runtime = Runtime()

response = agentcore_runtime.configure(
    entrypoint="agentcore_app_player_segmentation_agent.py",
    auto_create_execution_role=False,
    execution_role=agentcore_iam_role['Role']['Arn'],
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name
)

print("✅ AgentCore Runtime configured")
print(f"Configuration: {response}")

## Step 7: Launch Agent to AgentCore Runtime

Deploy the agent to the AgentCore Runtime environment, which handles containerization, ECR image creation, and serverless infrastructure provisioning. This process creates a production-ready endpoint for agent invocation with automatic scaling capabilities.

In [ ]:
print("🚀 Launching agent to AgentCore Runtime...")
launch_result = agentcore_runtime.launch()

print("✅ Agent launched successfully!")
print(f"Agent ARN: {launch_result.agent_arn}")
print(f"ECR URI: {launch_result.ecr_uri}")

## Step 8: Monitor Deployment Status

Monitor the agent deployment progress through the AgentCore Runtime status API. This polling mechanism tracks the containerization, infrastructure provisioning, and readiness validation phases until the agent endpoint becomes available for invocation.

In [ ]:
print("⏳ Checking deployment status...")

status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']

while status not in end_status:
    print(f"Status: {status}")
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']

print(f"✅ Final Status: {status}")

if status == 'READY':
    print("🎉 Agent is ready for invocation!")
else:
    print(f"❌ Deployment failed with status: {status}")

## Step 9: Test the Deployed Agent

Validate the deployed agent functionality by invoking it with a comprehensive player segmentation request. This test confirms that the agent can successfully access vector storage, perform semantic queries, and generate actionable business insights in the production environment.

In [ ]:
if status == 'READY':
    print("🧪 Testing deployed agent...")
    
    test_payload = {
        "prompt": "Analyze the player data and discover meaningful player segments for our game"
    }
    
    invoke_response = agentcore_runtime.invoke(test_payload)

else:
    print("⚠️ Cannot test agent - deployment not ready")

In [ ]:
import json

# Parse the response by joining all parts
content = invoke_response['response']
full_response = ''.join(content)
response_data = json.loads(full_response)
blocks = response_data['result']['content']
agent_text = next(b['text'] for b in blocks if 'text' in b)
print(agent_text)

## Step 10: Test with boto3 Client

Demonstrate direct agent invocation using the boto3 AgentCore client, which provides programmatic access for application integration. This approach enables session management and direct API calls for production applications that need to embed agent capabilities.

In [ ]:
import uuid

print("🔧 Testing with boto3 client...")

agentcore_client = boto3.client('bedrock-agentcore', region_name=region)
session_id = f"session-{uuid.uuid4().hex}"

boto3_response = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=launch_result.agent_arn,
    runtimeSessionId=session_id,
    payload=json.dumps({
        "prompt": "What are the key characteristics of high-value players in our dataset?"
    }),
    qualifier="DEFAULT"
)

print(f"Session ID: {session_id}\n")

response_body = boto3_response['response'].read()
response_data = json.loads(response_body)
blocks = response_data['result']['content']
agent_text = next(b['text'] for b in blocks if 'text' in b)

print("Agent Response:\n")
print(agent_text)

## Step 11: Display Agent ARN

Display the deployed agent ARN for use in applications.

In [ ]:
print(f"Player Segmentation Agent ARN: {launch_result.agent_arn}")

## Congratulations! 🎉

You have successfully:

✅ **Deployed your agent** to Amazon Bedrock AgentCore Runtime  
✅ **Created production infrastructure** with IAM roles and ECR repositories  
✅ **Tested agent deployment** using both SDK and boto3 invocation methods

### Key Concepts Learned:
- **AgentCore Runtime**: Production-ready serverless environment for AI agents
- **Session Isolation**: Security through dedicated execution environments
- **Multi-framework Support**: Compatible with various AI agent frameworks
- **Built-in Observability**: Integrated monitoring and debugging capabilities

Your agent is now deployed in a production-ready environment and ready for integration with applications.

---

## 🚀 Next Steps: Test Your Agent in the Frontend

Now that your agent is deployed, you can test it using the Streamlit frontend:

1. **Copy the Player Segmentation Agent ARN** from the output above (it looks like: `arn:aws:bedrock-agentcore:us-west-2:123456789012:agent/abc123...`)

2. **Open the Streamlit frontend** and navigate to the **Segment Discovery** tab

3. **Paste the Player Segmentation Agent ARN** into the agent configuration section

4. **Try asking the agent** to discover player segments.

5. **Save discovered segments** using the segment management features

The agent will use the vector embeddings from Lab 1 to provide contextual, data-driven segment recommendations!